# MSC Figure 4 — stateful stepped-drive hysteresis

**Drive-free and reproducible.** Upload the MSC model and its YAML configuration, execute every cell from top to bottom, and download one ZIP bundle at the end.

## Canonical protocol

- 100 parameter-generating seeds; base seed 7000.
- Activity drive is clamped at 0.0, 0.1, …, 1.0 on the up branch, then at 1.0, 0.9, …, 0.0 on the down branch. The repeated 1.0 plateau is retained deliberately: the first instance is the end of the up branch and the second is the beginning of the down branch.
- Each plateau lasts 4,200 s; the final 600 s provide the late-state readout.
- The complete stochastic state is propagated continuously between plateaus: S, M, granule positions and directions, target vectors, random-number-generator state, and the delayed-payload queue. **No plateau is reinitialised.**

## Outputs

| File | Role |
|---|---|
| `msc_feedback_hysteresis_raw.csv` | Per-seed, per-branch, per-drive late states and branch gaps |
| `msc_feedback_hysteresis_summary.csv` | Mean ± SEM branch trajectories |
| `msc_feedback_hysteresis_seed_statistics.csv` | True trapezoidal signed hysteresis areas, maximum gaps, and Wilcoxon tests |
| `msc_feedback_hysteresis_drive_gaps.csv` | Per-seed branch separations at each matched drive |
| `MANIFEST_figure4.json` | Exact run settings plus SHA-256 hashes of all inputs and outputs |

The companion R script should use the seed-level **trapezoidal signed areas**, not the mean branch gap, for Figure 4E.

## 1 — Environment

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy'], check=False)
import hashlib, importlib, json, math, os, shutil, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__)

## 2 — Upload model and YAML

Upload `msc_model_feedback.py` (or `msc_model_feedback_used.py`) and `config_msc_feedback.yaml` (or your corresponding YAML). The notebook records the SHA-256 hash of both files.

In [ ]:
from google.colab import files
WORK = Path('/content/msc_figure4'); WORK.mkdir(exist_ok=True)
print('Select the model .py file and the YAML configuration.')
uploaded = files.upload()
for name in uploaded: shutil.move(f'/content/{name}', WORK / name)
py_files = sorted(WORK.glob('*.py')); yaml_files = sorted(list(WORK.glob('*.yaml')) + list(WORK.glob('*.yml')))
assert py_files, 'No .py model file uploaded.'
assert yaml_files, 'No YAML configuration uploaded.'
MODEL = WORK / 'msc_model_feedback.py'
if not MODEL.exists():
    py_files[0].rename(MODEL)
    print(f'Renamed {py_files[0].name} -> msc_model_feedback.py')
CONFIG = yaml_files[0]
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
INPUT_SHA = {p.name: sha256(p) for p in (MODEL, CONFIG)}
for name, digest in INPUT_SHA.items(): print(digest[:16], name)
os.chdir(WORK); sys.path.insert(0, str(WORK))
import msc_model_feedback as msc
cfg = msc.load_config(str(CONFIG))
print('Model and configuration loaded successfully.')

## 3 — Canonical settings

Run the smoke test first. It uses 3 seeds and confirms that the stateful runner, output files and ZIP construction work. Then set `SMOKE = False` and restart the notebook from this cell onward; for a fully clean run, use **Runtime → Restart session and run all** after changing it.

The final canonical analysis uses 100 seeds, drives from 0.0 to 1.0, and exactly the 4,200-s / 600-s protocol described in the manuscript.

In [ ]:
SMOKE = True  # Set False only after the 3-seed test has completed successfully.
RUN = {
    'N_SEEDS': 100, 'BASE_SEED': 7000,
    'DRIVE_LEVELS': [round(x, 1) for x in np.arange(0.0, 1.01, 0.1)],
    'PLATEAU_SECONDS': 4200.0, 'LATE_WINDOW_SECONDS': 600.0,
    'VARIANT': 'full_msc_feedback',
    'STATE_PROPAGATED': True,
    'AREA_METHOD': 'numpy.trapezoid over matched drive levels',
    'AREA_SIGN': 'down minus up',
    'WILCOXON_ALTERNATIVE': 'greater'
}
OUT = WORK / 'figure4_output'
for sub in ['01_inputs', '02_raw', '03_derived', '04_r_input', '05_logs']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
shutil.copy2(MODEL, OUT/'01_inputs'/MODEL.name); shutil.copy2(CONFIG, OUT/'01_inputs'/CONFIG.name)
(OUT/'01_inputs'/'run_config.json').write_text(json.dumps(RUN, indent=2))
N_RUN = 3 if SMOKE else RUN['N_SEEDS']
print(json.dumps(RUN, indent=2)); print('Seeds to run:', N_RUN)

## 4 — Stateful MSC simulator

This implementation uses the same transport, docking, payload and feedback equations as the uploaded simulator. Unlike `simulate_one()`, it exposes the complete simulation state so that it can be passed from one drive plateau to the next. The active-synapse activity tag is clamped to the current drive at every timestep; all other state variables evolve normally.

In [ ]:
def initialise_state(p, rng, variant_name):
    variant = msc.MODEL_VARIANTS[variant_name]
    ns, ng, dm, L = int(p['Nsyn']), int(p['Ngran']), int(p['D_material']), float(p['L'])
    xsyn = np.linspace(L/(ns+1), L-L/(ns+1), ns)
    active = ns // 2
    theta = rng.normal(0.0, 1.0, size=(ns, dm)); theta[active, :] = 0.0
    u_true, q_g, strong = msc.make_material_pool(p, rng)
    if variant['mode'] == 'shuffled': u_used = u_true[rng.permutation(ng)]
    elif variant['mode'] == 'random': u_used = rng.normal(0.0, 1.25, size=u_true.shape)
    else: u_used = u_true.copy()
    return dict(rng=rng, xsyn=xsyn, active=active, theta=theta, u_used=u_used, q_g=q_g, strong=strong,
                gpos=rng.uniform(0.0, L, size=ng), gdir=rng.choice([-1.0, 1.0], size=ng),
                S=np.zeros(ns), M=np.full(ns, float(p['M_bas'])), pending=np.zeros((1, ns)))

def extend_pending(state, needed):
    if state['pending'].shape[0] < needed:
        extra = np.zeros((needed-state['pending'].shape[0], state['pending'].shape[1]))
        state['pending'] = np.vstack([state['pending'], extra])

def advance_plateau(state, p, variant_name, drive, plateau_seconds, late_seconds):
    variant = msc.MODEL_VARIANTS[variant_name]
    dt, L = float(p['dt']), float(p['L'])
    n_steps = int(round(plateau_seconds/dt)); late_steps = int(round(late_seconds/dt))
    delay_steps = max(1, int(round(float(p['delta_translation'])/dt)))
    extend_pending(state, n_steps + delay_steps + 2)
    ns, ng = int(p['Nsyn']), int(p['Ngran'])
    material_on, feedback_on = bool(variant['material']), bool(variant['feedback'])
    axes = msc.axes_from_variant(variant, int(p['D_material']))
    kn = float(p['K_M']) ** int(p['n_hill'])
    S_late, M_late = [], []
    for step in range(n_steps):
        state['S'] += state['pending'][0]
        state['pending'] = np.vstack([state['pending'][1:], np.zeros((1, ns))])
        state['S'] += (-state['S']/float(p['tau_S']))*dt
        state['S'] = np.clip(state['S'], 0.0, 1.0)
        if feedback_on:
            sn = state['S'] ** int(p['n_hill']); H = sn/(kn + sn + 1e-12)
            state['M'] += (-float(p['k_M'])*(state['M']-float(p['M_bas'])) + float(p['alphaM'])*H*(float(p['M_max'])-state['M']))*dt
            state['M'] = np.clip(state['M'], 0.0, 1.0)
        state['gpos'] += state['gdir']*float(p['v'])*dt + np.sqrt(2*float(p['D'])*dt)*state['rng'].standard_normal(ng)
        flips = state['rng'].random(ng) < float(p['p_switch']); state['gdir'][flips] *= -1.0
        below, above = state['gpos'] < 0.0, state['gpos'] > L
        state['gpos'][below] = -state['gpos'][below]; state['gpos'][above] = 2*L-state['gpos'][above]
        state['gdir'][below | above] *= -1.0
        for g in range(ng):
            near = np.flatnonzero(np.abs(state['xsyn']-state['gpos'][g]) <= float(p['epsilon']))
            for i in near:
                d2 = msc.material_distance2(state['u_used'][g], state['theta'][i], axes)
                phi_h = math.exp(-0.5*d2/float(p['ell_h'])**2) if material_on else 1.0
                phi_q = math.exp(-0.5*d2/float(p['ell_q'])**2) if material_on else 1.0
                T_i = drive if i == state['active'] else 0.0
                exponent = float(p['beta_T'])*T_i + (float(p['beta_M'])*state['M'][i] if feedback_on else 0.0)
                hazard = float(p['h0'])*math.exp(min(20.0, exponent))*phi_h
                if state['rng'].random() < 1.0-math.exp(-hazard*dt):
                    qig = float(state['q_g'][g]*phi_q)
                    if qig >= float(p['q_min']):
                        extend_pending(state, delay_steps+2)
                        state['pending'][delay_steps, i] += float(p['alphaS'])*qig
        if step >= n_steps-late_steps:
            S_late.append(float(state['S'][state['active']]))
            M_late.append(float(state['M'][state['active']]))
    return float(np.mean(S_late)), float(np.mean(M_late))

def run_one_seed(seed):
    p = msc.sample_params(seed, cfg); rng = np.random.default_rng(seed*100 + 17)
    state = initialise_state(p, rng, RUN['VARIANT'])
    rows = []
    up = RUN['DRIVE_LEVELS']; down = list(reversed(RUN['DRIVE_LEVELS']))
    for direction, levels in [('up', up), ('down', down)]:
        for drive in levels:
            S, M = advance_plateau(state, p, RUN['VARIANT'], drive, RUN['PLATEAU_SECONDS'], RUN['LATE_WINDOW_SECONDS'])
            rows.append({'seed': seed, 'direction': direction, 'activation_amp': drive,
                         'active_S_late': S, 'active_M_late': M})
    return rows, p
print('Stateful simulator defined.')

## 5 — Run 100 state-propagated protocols

A checkpoint is written after each seed. The full analysis evaluates 22 plateaus per seed—11 on the up branch and 11 on the down branch. The duplicate drive=1.0 plateau is intentional and preserves the stateful branch transition.

In [ ]:
rows, parameters = [], []
checkpoint = OUT/'02_raw'/'checkpoint_raw.csv'
t0 = time.time()
for j in range(N_RUN):
    seed = RUN['BASE_SEED'] + j
    seed_rows, p = run_one_seed(seed)
    rows.extend(seed_rows); parameters.append(p)
    pd.DataFrame(rows).to_csv(checkpoint, index=False)
    if (j+1) % max(1, min(5, N_RUN)) == 0:
        elapsed = time.time()-t0
        print(f'{j+1}/{N_RUN} seeds | {elapsed/60:.1f} min elapsed | ETA {elapsed/(j+1)*(N_RUN-j-1)/60:.1f} min', flush=True)
raw = pd.DataFrame(rows)
params = pd.DataFrame(parameters)
print('Raw rows:', len(raw), '| expected:', N_RUN*22)
assert len(raw) == N_RUN*22, 'Unexpected number of branch/drive records.'

## 6 — Derive trajectories, true signed areas and tests

The signed area is calculated separately for each seed as the trapezoidal integral of `down − up` across the eleven matched drive levels. Positive values mean that the down branch remained elevated after earlier high drive. This is the statistic tested by one-sided Wilcoxon signed-rank tests.

In [ ]:
summary = (raw.groupby(['direction','activation_amp'], as_index=False)
             .agg(S_mean=('active_S_late','mean'), S_sem=('active_S_late', lambda x: x.std(ddof=1)/np.sqrt(len(x))),
                  M_mean=('active_M_late','mean'), M_sem=('active_M_late', lambda x: x.std(ddof=1)/np.sqrt(len(x))),
                  n=('seed','nunique')))
wide = raw.pivot(index=['seed','activation_amp'], columns='direction', values=['active_S_late','active_M_late']).reset_index()
wide.columns = ['seed','activation_amp','M_down','M_up','S_down','S_up']
wide['S_gap'] = wide['S_down']-wide['S_up']; wide['M_gap'] = wide['M_down']-wide['M_up']
wide = wide.sort_values(['seed','activation_amp'])
stats_rows = []
for seed, g in wide.groupby('seed'):
    x = g.activation_amp.to_numpy()
    s_gap, m_gap = g.S_gap.to_numpy(), g.M_gap.to_numpy()
    stats_rows.append({'seed':seed, 'S_signed_area':float(np.trapezoid(s_gap, x)), 'M_signed_area':float(np.trapezoid(m_gap, x)),
                       'S_max_gap':float(np.max(s_gap)), 'M_max_gap':float(np.max(m_gap)),
                       'S_mean_gap':float(np.mean(s_gap)), 'M_mean_gap':float(np.mean(m_gap))})
seed_stats = pd.DataFrame(stats_rows)
test_rows=[]
for variable in ['S_signed_area','M_signed_area','S_max_gap','M_max_gap']:
    x=seed_stats[variable].dropna().to_numpy(); w,pv=wilcoxon(x, zero_method='wilcox', alternative='greater', method='approx')
    test_rows.append({'metric':variable, 'n':len(x), 'mean':float(np.mean(x)), 'median':float(np.median(x)),
                      'sd':float(np.std(x,ddof=1)), 'wilcoxon_W':float(w), 'one_sided_p':float(pv)})
tests=pd.DataFrame(test_rows)
delta_S = float(wide.groupby('activation_amp').S_gap.mean().max()); delta_M = float(wide.groupby('activation_amp').M_gap.mean().max())
print('Maximum mean branch separations: ΔS =', round(delta_S,3), '| ΔM =', round(delta_M,3))
display(tests)

## 7 — Save raw data, R inputs, manifest and ZIP

The R script reads the two CSV files copied to `04_r_input/`. It can also read the seed-level areas and tests from that same directory. The ZIP is the reproducibility unit for Figure 4.

In [ ]:
D_RAW=OUT/'02_raw'; D_DER=OUT/'03_derived'; D_R=OUT/'04_r_input'
raw.to_csv(D_RAW/'msc_feedback_hysteresis_raw.csv', index=False)
summary.to_csv(D_RAW/'msc_feedback_hysteresis_summary.csv', index=False)
params.to_csv(D_RAW/'parameter_realizations.csv', index=False)
wide.to_csv(D_DER/'msc_feedback_hysteresis_drive_gaps.csv', index=False)
seed_stats.to_csv(D_DER/'msc_feedback_hysteresis_seed_statistics.csv', index=False)
tests.to_csv(D_DER/'msc_feedback_hysteresis_wilcoxon_tests.csv', index=False)
for source, name in [(D_RAW,'msc_feedback_hysteresis_raw.csv'),(D_RAW,'msc_feedback_hysteresis_summary.csv'),
                     (D_DER,'msc_feedback_hysteresis_drive_gaps.csv'),(D_DER,'msc_feedback_hysteresis_seed_statistics.csv'),
                     (D_DER,'msc_feedback_hysteresis_wilcoxon_tests.csv')]: shutil.copy2(source/name, D_R/name)
checks={'seed_count':int(raw.seed.nunique()), 'expected_seed_count':N_RUN, 'n_drive_levels':len(RUN['DRIVE_LEVELS']),
        'drive_min':float(raw.activation_amp.min()), 'drive_max':float(raw.activation_amp.max()),
        'rows':int(len(raw)), 'expected_rows':int(N_RUN*22), 'delta_S':delta_S, 'delta_M':delta_M}
pd.DataFrame([checks]).to_csv(OUT/'05_logs'/'Figure4_run_checks.csv', index=False)
(OUT/'README.md').write_text('# MSC Figure 4 reproducible bundle\n\nRun protocol, source inputs, raw per-seed trajectories, derived true trapezoidal signed areas, statistical tests and R-ready tables are included. The figure uses a stateful 0.0–1.0 stepped-drive protocol with 100 seeds.\n')
manifest={'figure':'Figure 4','generated_utc':pd.Timestamp.utcnow().isoformat(),'smoke_run':SMOKE,'run_config':RUN,'checks':checks,'input_sha256':INPUT_SHA,
          'output_sha256':{str(p.relative_to(OUT)):sha256(p) for p in sorted(OUT.rglob('*')) if p.is_file()}}
(OUT/'MANIFEST_figure4.json').write_text(json.dumps(manifest,indent=2))
zip_path=Path('/content/MSC_Figure4_reproducible_bundle.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(OUT.parent))
print('ZIP:',zip_path.name, f'({zip_path.stat().st_size/1e6:.2f} MB)')
print(checks)
files.download(str(zip_path))